# Synctra — All-in-One Colab

Runs the full Synctra stack inside a single Colab GPU runtime:

| Component | Port | Source |
|---|---|---|
| Trained NLP tool router | 8001 | `tool/colab_course_import_agent_server.py` + `tool/nlp_tool_calling_agent.py` |
| Qwen course-import LLM | 8001 (same) | same file |
| FastAPI backend (uvicorn) | 8000 | `backend/` |
| Cloudflared tunnel | — | exposes port 8000 to your Flutter app |

### Before running
- **Runtime → Change runtime type → T4 GPU** (or better).
- Have your GitHub repo URL ready, or be ready to upload files manually.

### Run cells top-to-bottom. The final cell blocks; keep the runtime open while your app is in use.

## 1. Install dependencies

In [ ]:
!pip -q install torch transformers accelerate datasets sentencepiece
!pip -q install fastapi uvicorn pydantic pydantic-settings httpx python-dotenv
!pip -q install beautifulsoup4 supabase
print('Dependencies installed.')

## 2. Get the repo into Colab

**Option A** — clone from GitHub (recommended). Replace the URL with your fork if needed.

In [ ]:
REPO_URL = 'https://github.com/rpham03/synctraCapstone.git'
REPO_DIR = '/content/syntra'

import os, subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone', REPO_URL, REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
print('Repo ready at', REPO_DIR)
!ls $REPO_DIR/tool/ | head

**Option B** — if your repo is private, run this cell instead and upload `synctra_capstone.zip` from the Files panel.

In [ ]:
# Uncomment to use upload instead of git clone:
#
# from google.colab import files
# uploaded = files.upload()  # pick synctra_capstone.zip
# import zipfile, shutil, os
# shutil.rmtree('/content/syntra', ignore_errors=True)
# with zipfile.ZipFile(list(uploaded.keys())[0]) as zf:
#     zf.extractall('/content/syntra')
# print(os.listdir('/content/syntra'))

## 3. Train the NLP tool router  *(skip if you already have a model)*

Takes ~5–15 min on a T4. Outputs `/content/syntra_tool_router/`.

Skip this cell if you previously saved the model to Drive — the next section pulls it from there.

In [ ]:
!python /content/syntra/tool/one_click_train_nlp_router_colab.py

### 3b. (Optional) Persist the trained model to Google Drive

Colab wipes `/content/` between sessions. Save the model to Drive once, then restore it later instead of retraining.

In [ ]:
# Mount Drive and copy the trained model.
from google.colab import drive
import os, shutil
drive.mount('/content/drive')

DRIVE_MODEL_DIR = '/content/drive/MyDrive/syntra_tool_router'
LOCAL_MODEL_DIR = '/content/syntra_tool_router'

if os.path.isdir(LOCAL_MODEL_DIR) and not os.path.isdir(DRIVE_MODEL_DIR):
    shutil.copytree(LOCAL_MODEL_DIR, DRIVE_MODEL_DIR)
    print('Saved model to Drive at', DRIVE_MODEL_DIR)
elif os.path.isdir(DRIVE_MODEL_DIR) and not os.path.isdir(LOCAL_MODEL_DIR):
    shutil.copytree(DRIVE_MODEL_DIR, LOCAL_MODEL_DIR)
    print('Restored model from Drive')
else:
    print('Local model present:', os.path.isdir(LOCAL_MODEL_DIR),
          '| Drive model present:', os.path.isdir(DRIVE_MODEL_DIR))

## 4. Start the model server (port 8001) in the background

Loads Qwen (course import + ai_agent fallback) and the trained classifier (chat router). Wait ~60 seconds for the model to download/load before continuing.

In [ ]:
import subprocess, time, os, sys, urllib.request, json

TOOL_DIR = '/content/syntra/tool'
MODEL_DIR = '/content/syntra_tool_router'

assert os.path.isdir(MODEL_DIR), f'No trained model at {MODEL_DIR}. Train it first or restore from Drive.'

model_server_log = open('/content/model_server.log', 'wb')
model_server = subprocess.Popen(
    [
        sys.executable, f'{TOOL_DIR}/colab_course_import_agent_server.py',
        '--host', '127.0.0.1',
        '--port', '8001',
        '--tunnel', 'none',
        '--nlp-router-model-dir', MODEL_DIR,
        '--nlp-agent-path', TOOL_DIR,
    ],
    stdout=model_server_log,
    stderr=subprocess.STDOUT,
)
print('Model server PID:', model_server.pid)

for attempt in range(120):
    time.sleep(2)
    try:
        with urllib.request.urlopen('http://127.0.0.1:8001/health', timeout=2) as r:
            data = json.loads(r.read())
        print('Model server up:', json.dumps(data, indent=2))
        break
    except Exception:
        if attempt % 10 == 0:
            print(f'... waiting for model server ({attempt*2}s)')
else:
    raise RuntimeError('Model server did not come up. Check /content/model_server.log')

## 5. Configure the Synctra backend

- Points `OLLAMA_HOST` at the local model server.
- Registers the `chat_colab` route in `main.py` if not already wired up.

In [ ]:
import os, pathlib, re

BACKEND_DIR = pathlib.Path('/content/syntra/backend')
os.environ['OLLAMA_HOST'] = 'http://127.0.0.1:8001'

# Optional: write a .env so settings.py also picks it up.
env_file = BACKEND_DIR / '.env'
env_content = env_file.read_text() if env_file.exists() else ''
if 'OLLAMA_HOST' not in env_content:
    env_file.write_text(env_content + ('\n' if env_content else '') + 'OLLAMA_HOST=http://127.0.0.1:8001\n')
    print('Wrote OLLAMA_HOST into', env_file)

# Wire chat_colab into main.py if needed.
main_py = BACKEND_DIR / 'app/main.py'
src = main_py.read_text()
if 'chat_colab' not in src:
    src = re.sub(
        r'(from app\.api\.v1\.routes import [^\n]+)',
        r'\1, chat_colab',
        src,
        count=1,
    )
    src += (
        '\napp.include_router(chat_colab.router, prefix="/api/v1/chat-colab", tags=["chat-colab"])\n'
    )
    main_py.write_text(src)
    print('Registered chat_colab route in main.py')
else:
    print('chat_colab already registered')

# Install the backend requirements.
!pip -q install -r {BACKEND_DIR}/requirements.txt

## 6. Open a cloudflared tunnel for the backend (port 8000)

The printed `https://*.trycloudflare.com` URL is what you point your Flutter app at.

In [ ]:
import sys
sys.path.insert(0, '/content/syntra/tool')
from colab_course_import_agent_server import start_cloudflared_tunnel
import time
start_cloudflared_tunnel(8000)
time.sleep(4)
print('Cloudflared started. Copy the trycloudflare.com URL above into your Flutter app.')

## 7. Launch the FastAPI backend (uvicorn on port 8000)

This cell **blocks** — it must stay running for your app to work. To stop, click the stop button on this cell.

In [ ]:
%cd /content/syntra/backend
!uvicorn app.main:app --host 0.0.0.0 --port 8000

## 8. (Optional) Quick smoke test — run from a *separate* Colab cell or local terminal

While the uvicorn cell above keeps running, you can test from another browser tab:

```bash
curl https://YOUR_TUNNEL.trycloudflare.com/api/v1/chat-colab/health | jq

curl -X POST https://YOUR_TUNNEL.trycloudflare.com/api/v1/chat-colab/ \
  -H 'Content-Type: application/json' \
  -d '{"message":"what homework is due this week"}' | jq
```

Expected: the classifier routes to `get_tasks`, the backend executes it, and you get a JSON response.

If something fails, check `/content/model_server.log` for model server errors.